# Universal Model Evaluation Notebook

This notebook evaluates the best-performing article-level BiLSTM classifier on a custom dataset. 
It calculates classification accuracy, model confidence, and standard readability metrics (Flesch, Wiener Sachtextformel).

In [1]:
import torch
import torch.nn as nn
import pandas as pd
import json
import os
import spacy
import re
from collections import Counter
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, balanced_accuracy_score
from sklearn.model_selection import train_test_split
import textstat
import numpy as np
from tqdm import tqdm

# --- CONFIGURATION ---
# Path to the dataset you want to test (JSON format with 'ls_text' and 'as_text')
DATASET_PATH = "../results/lebenshilfe_dataset.json"

# Path to the saved model state
MODEL_PATH = "../results/lstm_article_sim_0.80_to_0.98.pt"

# Source file for vocabulary reconstruction (must match the model's training data)
VOCAB_SOURCE_CSV = "../results/information_loss_analysis_cleaned.csv"
VOCAB_SIM_RANGE = (0.8, 0.98)

# Model hyperparameters (must match training)
MAX_SEQ_LEN = 512
EMBED_DIM = 128
HIDDEN_DIM = 128

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

Using device: cpu


## 1. Class Definitions

In [2]:
class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_dim * 2, 1)
        self.dropout = nn.Dropout(0.4)
        
    def forward(self, x):
        embedded = self.dropout(self.embedding(x))
        _, (hidden, _) = self.lstm(embedded)
        hidden = torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)
        return self.fc(self.dropout(hidden))

class Vocab:
    def __init__(self, sentences, max_size=25000, min_freq=3):
        counter = Counter()
        for sent in sentences:
            counter.update(sent)
        
        self.itos = ["<pad>", "<unk>"]
        self.stoi = {"<pad>": 0, "<unk>": 1}
        
        for token, freq in counter.most_common(max_size):
            if freq >= min_freq:
                self.stoi[token] = len(self.itos)
                self.itos.append(token)
                
    def __len__(self):
        return len(self.itos)
    
    def encode(self, tokens):
        return [self.stoi.get(t, self.stoi["<unk>"]) for t in tokens]

## 2. Environment Setup
Reconstructing the vocabulary and loading the model.

In [3]:
def build_original_vocab():
    print(f"Reconstructing vocab from {VOCAB_SOURCE_CSV}...")
    df = pd.read_csv(VOCAB_SOURCE_CSV)
    mask = (df["semantic_similarity_8192"] >= VOCAB_SIM_RANGE[0]) & (df["semantic_similarity_8192"] <= VOCAB_SIM_RANGE[1])
    df_filtered = df[mask]
    
    nlp = spacy.blank("de")
    X, y = [], []
    
    for _, row in df_filtered.iterrows():
        ls_tokens = [t.text.lower() for t in nlp(str(row["ls_text"])) if not t.is_space]
        if len(ls_tokens) >= 10:
            X.append(ls_tokens)
            y.append(1)
        as_tokens = [t.text.lower() for t in nlp(str(row["as_text"])) if not t.is_space]
        if len(as_tokens) >= 10:
            X.append(as_tokens)
            y.append(0)
    
    X_train_val, _, y_train_val, _ = train_test_split(X, y, test_size=0.15, random_state=42, stratify=y)
    X_train, _, _, _ = train_test_split(X_train_val, y_train_val, test_size=0.15, random_state=42, stratify=y_train_val)
    
    return Vocab(X_train)

vocab = build_original_vocab()
print(f"Vocab size: {len(vocab)}")

model = BiLSTMClassifier(len(vocab), EMBED_DIM, HIDDEN_DIM).to(DEVICE)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()
print("Model loaded successfully.")

Reconstructing vocab from ../results/information_loss_analysis_cleaned.csv...
Vocab size: 7027
Model loaded successfully.


## 3. Evaluation Logic

In [4]:
nlp = spacy.blank("de")

def predict(text):
    tokens = [t.text.lower() for t in nlp(text) if not t.is_space]
    encoded = vocab.encode(tokens)[:MAX_SEQ_LEN]
    padded = encoded + [0] * (MAX_SEQ_LEN - len(encoded))
    tensor = torch.tensor([padded], dtype=torch.long).to(DEVICE)
    
    with torch.no_grad():
        output = model(tensor).squeeze()
        prob = torch.sigmoid(output).item()
        pred = 1 if prob > 0.5 else 0
    return pred, prob

def run_evaluation(path):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    print(f"Evaluating {len(data)} pairs from {path}...")
    
    results = []
    for item in tqdm(data):
        ls_text = item.get("ls_text", "")
        as_text = item.get("as_text", "")
        
        if not ls_text or not as_text: continue
        
        ls_pred, ls_prob = predict(ls_text)
        as_pred, as_prob = predict(as_text)
        
        results.append({
            "LS_ID": item.get("ls_filename", "N/A"),
            "AS_ID": item.get("as_filename", "N/A"),
            "LS_Pred": "Simple" if ls_pred == 1 else "Normal",
            "LS_Conf": ls_prob if ls_pred == 1 else 1 - ls_prob,
            "AS_Pred": "Simple" if as_pred == 1 else "Normal",
            "AS_Conf": as_prob if as_pred == 1 else 1 - as_prob,
            "LS_Flesch": textstat.flesch_reading_ease(ls_text),
            "AS_Flesch": textstat.flesch_reading_ease(as_text),
            "LS_Wiener": textstat.wiener_sachtextformel(ls_text, 1),
            "AS_Wiener": textstat.wiener_sachtextformel(as_text, 1),
            "Correct": (ls_pred == 1 and as_pred == 0)
        })
    return pd.DataFrame(results)

## 4. Run & Results

In [5]:
from sklearn.metrics import accuracy_score, balanced_accuracy_score

# 1. Run the evaluation
df_res = run_evaluation(DATASET_PATH)

# 2. Prepare data for standard classification metrics
y_true = [1] * len(df_res) + [0] * len(df_res) # LS=1, AS=0
y_pred = list(df_res["LS_Pred"].map({"Simple": 1, "Normal": 0})) + list(df_res["AS_Pred"].map({"Simple": 1, "Normal": 0}))

acc = accuracy_score(y_true, y_pred)
bacc = balanced_accuracy_score(y_true, y_pred)

print("\n--- CLASSIFICATION METRICS ---")
print(f"Overall Accuracy: {acc*100:.2f}%")
print(f"Balanced Accuracy: {bacc*100:.2f}%")
print(f"Perfect Pair Match: {df_res['Correct'].sum()} / {len(df_res)} ({df_res['Correct'].mean()*100:.1f}%) - (Both LS & AS correct)")

print("\n--- READABILITY METRICS ---")
print(f"Avg LS Flesch: {df_res['LS_Flesch'].mean():.2f} (Higher = Easier, target LS: >80)")
print(f"Avg AS Flesch: {df_res['AS_Flesch'].mean():.2f}")
print(f"Flesch Gap: {df_res['LS_Flesch'].mean() - df_res['AS_Flesch'].mean():.2f}")

print(f"\nAvg LS Wiener: {df_res['LS_Wiener'].mean():.2f} (Lower = Easier, target LS: <6)")
print(f"Avg AS Wiener: {df_res['AS_Wiener'].mean():.2f}")

print("\n--- PREDICTION COUNTS ---")
print(f"LS correctly identified as Simple: {df_res[df_res['LS_Pred']=='Simple'].shape[0]} / {len(df_res)}")
print(f"AS correctly identified as Normal: {df_res[df_res['AS_Pred']=='Normal'].shape[0]} / {len(df_res)}")

df_res.head(10)

Evaluating 49 pairs from ../results/lebenshilfe_dataset.json...


100%|██████████| 49/49 [00:01<00:00, 31.02it/s]


--- CLASSIFICATION METRICS ---
Overall Accuracy: 90.82%
Balanced Accuracy: 90.82%
Perfect Pair Match: 40 / 49 (81.6%) - (Both LS & AS correct)

--- READABILITY METRICS ---
Avg LS Flesch: 66.40 (Higher = Easier, target LS: >80)
Avg AS Flesch: 43.29
Flesch Gap: 23.11

Avg LS Wiener: 5.19 (Lower = Easier, target LS: <6)
Avg AS Wiener: 9.07

--- PREDICTION COUNTS ---
LS correctly identified as Simple: 46 / 49
AS correctly identified as Normal: 43 / 49


,LS_ID,AS_ID,LS_Pred,LS_Conf,AS_Pred,AS_Conf,LS_Flesch,AS_Flesch,LS_Wiener,AS_Wiener,Correct
0,ILS_CAU_Geologiemuseum AD002 Prüfer Mail.docx,20241106-PM-Aktionstag-CAU und StK-an PS_neu.docx,Simple,0.989295,Normal,0.996525,79.500501,40.254626,3.712667,9.114980,True
1,ILS LB MmB Positionspapier Arbeit 004.docx,Positionspapier Originaltext.docx,Simple,0.988830,Normal,0.996377,55.088169,43.546475,6.327928,8.610855,True
2,ILS07 Einwilligung KS DS001 Prüfer.docx,07 BwH KS-Einwilligung-2024-10-16-19-16-06.rtf,Simple,0.996818,Normal,0.997059,64.231232,25.704167,4.967102,10.182337,True
3,ILS08 BwH Entbindung Schweigepflicht AD002 pru...,08 BwH Schweigepflichtentbindung - allgemein.odt,Normal,0.995161,Normal,0.994977,63.613553,37.946149,5.883622,10.205522,False
4,ILS09 BwH Einverständniserklärung AD001 prü...,09 BwH Einverständnis Bericht § 160 StGB (Bri...,Normal,0.991792,Normal,0.996686,62.015455,41.553571,6.394809,10.545800,False
5,ILS06 BwH Info KSKS AD001 prüfen.docx,06 BwH KSKS - Handreichung-2024-10-16-16-47-39...,Simple,0.974489,Normal,0.998166,70.079042,44.984615,4.940126,8.687516,True
6,ILS_IBA_Fischbeker_Reethen_AD002.docx,IBA Hamburg_Fischbeker Reethen_Web_Leichte Spr...,Simple,0.983755,Normal,0.995535,66.869211,43.561296,5.273193,9.492234,True
7,ILS FRAGEN Podium - Parlamentarischer Abend - ...,FRAGEN Podium - Parlamentarischer Abend - Stan...,Simple,0.613869,Normal,0.996668,55.744145,49.247619,6.932163,7.741518,True
8,ILS KIWA_Bedarfsumfrage AD002 Prüfer.docx,Bedarfsumfrage KIWA.docx,Simple,0.993423,Simple,0.972985,69.353739,60.737883,3.999443,5.988123,False
9,ILS Pinneberg Satzung Behindertenbeirat 001 AD...,Satzung der Stadt Pinneberg für den Behindert...,Simple,0.995335,Normal,0.996155,69.363783,46.619261,4.909310,8.809188,True
